# Phase 6 — vérification

Le contrôle reconstruit le suivi, puis vérifie les totaux, les devises, la concentration de chaque mois, les plafonds, et l'accord avec les métriques de la phase 5. Il ne fixe ni le nombre de titres, ni le mois du rapport.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import (
    CURRENCY_WEIGHT_CAP,
    ISSUER_WEIGHT_CAP,
    MAX_POSITIONS,
    MIN_POSITIONS,
    SECTOR_WEIGHT_CAP,
    ZONE_WEIGHT_CAP,
)

from src.portfolio_construction.build_portfolios import (
    METRICS_FILE,
    MONTHLY_RETURNS_FILE,
    STRATEGY_IDS,
    WEIGHTS_FILE,
)
from src.portfolio_monitoring.build_monitoring import (
    POSITION_WEIGHT_FLOOR,
    TOP_POSITIONS,
    WEIGHT_SUM_TOLERANCE,
    build_monitoring,
)

written = build_monitoring()
report = pd.read_csv(written["report"])
sectors = pd.read_csv(written["sectors"])
zones = pd.read_csv(written["zones"])
exposure = pd.read_csv(written["exposure"])
history = pd.read_csv(written["concentration_history"])
limits = pd.read_csv(written["limits"])
concentration = pd.read_csv(written["concentration"])
top10 = pd.read_csv(written["top10"])
holdings = pd.read_csv(written["holdings"], dtype={"ticker": str, "yahoo_symbol": str})
weights = pd.read_csv(WEIGHTS_FILE, dtype={"ticker": str, "yahoo_symbol": str})
metrics = pd.read_csv(METRICS_FILE)

assert set(report["strategy"]) == set(STRATEGY_IDS)
assert report["report_month"].nunique() == 1
assert report["report_month"].eq(report["window_end"]).all()
assert holdings["weight"].ge(0).all()
assert not holdings["sector"].eq("").any()
assert not holdings["zone"].eq("").any()
assert not holdings["exposure"].eq("").any()

for strategy in STRATEGY_IDS:
    report_month = str(report["report_month"].iloc[0])
    current = weights.loc[weights["strategy"].eq(strategy) & weights["month"].astype(str).eq(report_month)]
    total = float(current["weight"].sum())
    assert abs(total - 1) <= WEIGHT_SUM_TOLERANCE
    for table in (sectors, zones, exposure):
        grouped = float(table.loc[table["strategy"].eq(strategy), "weight"].sum())
        assert abs(grouped - 1) <= 1e-6
    labels = set(exposure.loc[exposure["strategy"].eq(strategy), "exposure"])
    assert labels.isdisjoint({"Actions", "Obligations", "ETF"})
    published = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    monitored = concentration.loc[concentration["strategy"].eq(strategy)].iloc[0]
    assert int(monitored["names"]) == int(published["names"])
    assert abs(monitored["maximum_weight"] - published["maximum_weight"]) <= 1e-6
    assert abs(monitored["effective_holdings"] - published["effective_holdings"]) <= 1e-6
    assert abs(monitored["hhi"] * monitored["effective_holdings"] - 1) <= 1e-6
    assert abs(monitored["concentration_multiple"] - monitored["hhi"] * monitored["names"]) <= 1e-6
    assert monitored["invested_names"] <= monitored["names"]
    if strategy != "equal_weight":
        assert 25 <= int(monitored["invested_names"]) <= 40
    ranks = top10.loc[top10["strategy"].eq(strategy), "rank"].astype(int).tolist()
    assert ranks == list(range(1, len(ranks) + 1))
    assert 1 <= len(ranks) <= TOP_POSITIONS
    assert top10.loc[top10["strategy"].eq(strategy), "weight"].gt(POSITION_WEIGHT_FLOOR).all()
    snap = history.loc[
        history["strategy"].eq(strategy) & history["month"].astype(str).eq(report_month)
    ]
    assert len(snap) == 1
    assert abs(float(snap["hhi"].iloc[0]) - float(monitored["hhi"])) <= 1e-12
    watched = limits.loc[
        limits["strategy"].eq(strategy) & limits["month"].astype(str).eq(report_month)
    ].iloc[0]
    published_row = report.loc[report["strategy"].eq(strategy)].iloc[0]
    # The report month starts with drifted weights: a cap can be passed, and the watch must say so.
    for flag, value, cap in (
        ("issuer_over_cap", float(current["weight"].max()), ISSUER_WEIGHT_CAP),
        ("sector_over_cap", float(sectors.loc[sectors["strategy"].eq(strategy), "weight"].max()), SECTOR_WEIGHT_CAP),
        ("zone_over_cap", float(zones.loc[zones["strategy"].eq(strategy), "weight"].max()), ZONE_WEIGHT_CAP),
        ("currency_over_cap", float(exposure.loc[exposure["strategy"].eq(strategy), "weight"].max()), CURRENCY_WEIGHT_CAP),
    ):
        assert int(watched[flag]) == int(value > cap + 1e-8), (strategy, flag)
    assert str(watched["largest_sector"]) == str(published_row["largest_sector"])
    assert abs(float(watched["largest_sector_weight"]) - float(published_row["largest_sector_weight"])) <= 1e-8
    assert abs(float(watched["largest_zone_weight"]) - float(published_row["largest_zone_weight"])) <= 1e-8
    assert abs(float(watched["largest_currency_weight"]) - float(published_row["largest_currency_weight"])) <= 1e-8

equal = concentration.loc[concentration["strategy"].eq("equal_weight")].iloc[0]
assert equal["concentration_multiple"] + 1e-9 >= 1
assert int(equal["invested_names"]) == int(equal["names"])

path = pd.read_csv(written["path"])
contributions = pd.read_csv(written["contributions"])
monthly = pd.read_csv(MONTHLY_RETURNS_FILE)
for strategy in STRATEGY_IDS:
    series = path.loc[path["strategy"].eq(strategy)].sort_values("month")
    official = monthly.loc[monthly["strategy"].eq(strategy)].sort_values("month")
    assert list(series["month"]) == list(official["month"])
    assert abs(series["portfolio_return"] - official["portfolio_return"]).max() <= 1e-8
    wealth = float((1 + series["portfolio_return"]).prod())
    assert abs(series["cumulative_wealth"].iloc[-1] - wealth) <= 1e-8
    assert series["drawdown"].max() <= 1e-12
    grouped = contributions.loc[contributions["strategy"].eq(strategy)]
    summed = grouped.groupby("month")["month_contribution"].sum()
    compared = series.set_index("month")["portfolio_return"]
    assert abs(summed.reindex(compared.index) - compared).max() <= 1e-6
    assert list(history.loc[history["strategy"].eq(strategy), "month"].astype(str)) == list(series["month"].astype(str))
    assert list(limits.loc[limits["strategy"].eq(strategy), "month"].astype(str)) == list(series["month"].astype(str))

flags = ["issuer_over_cap", "sector_over_cap", "zone_over_cap", "currency_over_cap"]
assert limits[flags].isin([0, 1]).all().all()
from src.config import IPS_SECTOR_WEIGHT_CAP
from src.portfolio_monitoring.build_monitoring import REBALANCE_MONTHS as DECISION_MONTHS

first_month = str(limits["month"].min())
expected_decision = limits["month"].astype(str).map(
    lambda value: value == first_month or int(value[5:7]) in DECISION_MONTHS
)
assert limits["rebalance_month"].astype(bool).tolist() == expected_decision.tolist()
# At a decision the caps hold. Between two decisions, drift can pass them.
assert int(limits.loc[limits["rebalance_month"].eq(1), flags].to_numpy().sum()) == 0
assert limits["sector_over_ips_cap"].astype(bool).tolist() == (limits["largest_sector_weight"] > IPS_SECTOR_WEIGHT_CAP + 1e-8).tolist()
feasibility = pd.read_csv(written["feasibility"])
total_row = feasibility.loc[feasibility["sector"].eq("TOTAL")].iloc[0]
assert abs(float(total_row["capacity_ips"]) - float(feasibility.loc[feasibility["sector"].ne("TOTAL"), "capacity_ips"].sum())) <= 1e-12
if float(total_row["capacity_ips"]) < 1 - 1e-9:
    # No fully invested book can respect the written sector limit: every book-month must show the breach.
    assert limits["sector_over_ips_cap"].eq(1).all()
assert limits["positions_outside_band"].isin([0, 1]).all()
assert int(limits.loc[limits["strategy"].eq("equal_weight"), "positions_outside_band"].sum()) == 0
for strategy in ("minimum_volatility", "maximum_sharpe"):
    book = limits.loc[limits["strategy"].eq(strategy)]
    expected = ~book["invested_names"].between(MIN_POSITIONS, MAX_POSITIONS)
    assert book["positions_outside_band"].astype(bool).tolist() == expected.tolist()

from src.config import QUARTERLY_TURNOVER_CAP
from src.portfolio_monitoring.build_monitoring import ANNUAL_TURNOVER_CAP, REBALANCE_MONTHS

turnover = pd.read_csv(written["turnover"])
assert set(turnover["strategy"]) == set(STRATEGY_IDS)
for strategy in STRATEGY_IDS:
    book = turnover.loc[turnover["strategy"].eq(strategy)].sort_values("month")
    assert book["month"].map(lambda value: int(str(value)[5:7])).isin(REBALANCE_MONTHS).all()
    above_quarter = book["gross_turnover"] > QUARTERLY_TURNOVER_CAP + 1e-8
    assert book["over_quarterly_cap"].astype(bool).tolist() == above_quarter.tolist()
    complete = book["trailing_annual_turnover"].notna()
    assert book.loc[~complete, "over_annual_cap"].eq(0).all()
    above_year = book.loc[complete, "trailing_annual_turnover"] > ANNUAL_TURNOVER_CAP + 1e-8
    assert book.loc[complete, "over_annual_cap"].astype(bool).tolist() == above_year.tolist()
from src.portfolio_monitoring.build_monitoring import shared_asset_returns

asset_returns = shared_asset_returns()
for strategy in STRATEGY_IDS:
    sample = turnover.loc[turnover["strategy"].eq(strategy)].sort_values("month").iloc[2]
    sample_month = str(sample["month"])
    book_weights = weights.loc[weights["strategy"].eq(strategy)]
    symbols = book_weights["yahoo_symbol"].drop_duplicates()
    previous_month = max(
        month for month in book_weights["month"].astype(str).unique() if month < sample_month
    )

    def weight_vector(month):
        rows = book_weights.loc[book_weights["month"].astype(str).eq(month)]
        return rows.set_index("yahoo_symbol")["weight"].reindex(symbols).fillna(0.0)

    # What the market left: the previous month's book, moved by that month's returns.
    grown = weight_vector(previous_month) * (1 + asset_returns.loc[previous_month, symbols])
    left = grown / grown.sum()
    rebuilt = float((weight_vector(sample_month) - left).abs().sum())
    assert abs(rebuilt - float(sample["gross_turnover"])) <= 1e-8, strategy
    # A book that looks idle on paper still trades once the market has moved.
    if strategy == "equal_weight":
        assert float(sample["gross_turnover"]) > 1e-4

from src.portfolio_monitoring.dashboard import DASHBOARD_FILE, SECTION_TITLES

dashboard_html = DASHBOARD_FILE.read_text(encoding="utf-8")
for title in SECTION_TITLES:
    assert title.split()[0] in dashboard_html
assert str(report["report_month"].iloc[0]) in dashboard_html

print(
    f"Rapport : {report['report_month'].iloc[0]}. "
    f"Stratégies : {report['strategy'].nunique()}."
)
print("Résultat : PASS")


Rapport : 2026-10. Stratégies : 3.
Résultat : PASS
